# Autoencoders (2-23-26)

## Introduction
* Many of the models we have used over the past 1.5 semesters are supervised, they require labeled training data
* However, most real world data is unlabeled or can be labeled with significant time and effort
* Autoencoders are an unsupervised machine learning algorithm for:
    * Representation learning
    * Dimensionality Reduction
    * Denoising
    * Pre-training Deep Networks
    * Generative Modeling (with tweaks covered on Wednesday)

## What is an Autoencoder?
* Consider a data set $x$
* An autoencoder consists of two functions:
    * The **encoder**: $z = f_\theta(x)$
    * The **decoder**: $\hat{x} = g_\phi(z)$
    * The encoder maps the inputs to a _latent representation_ and the decoder maps from the _latent representation_ to a reconstruction of the data
    * The model minimizes the reconstruction error (usually): $L(x,\hat{x}) = ||x-\hat{x}||^2$

## Autoencoders and PCA
* If:
    * The encoder and decoder are linear
    * The loss is the MSE
    * No nonlinearities in the model
* Then the autoeconder learns the same subsapce as Principal Component Analysis (PCA). PCA is a special case of autoencoders.

## Undercomplete and Overcomplete
* If $dim(z) < dim(x)$ then model is undercomplete and it forces compression. This encourages the model to learn useful structures
* If $dim(z) \geq dim(x)$ then the model is overcomplete and the model can learn identity mapping (basically learning nothing). Can fix this by adding regularization or adding noise.

## Types of Autoencoders

## Denoising
* Instead of feeding a clean input, $x$, feed a corrupted input $\bar{x} = x + \epsilon$ and train the model to reconstruct the original data: $L = ||x-g(f(\bar{x}))||^2$.$

### Variational Autoencoders (VAEs) (More on this Wednesday)
* Instead of encoding and decoding using functions the encoder and decoder are distributions.
* The latent space becomes continuous and structured. Foundational model for generative modeling.

### Convolutional Autoencoders
* Both encoder and decoder are made with convolutional layers instead of linear layers.

## Limitations
* Reconstruction is not the same as semantic understanding
* Can memorize without learning structure
* Latent space is not guaranteed to be interpretable
* Sensitive to architecture and regularization

## PyTorch Implementation - Anomaly Detection

In [1]:
#############
## IMPORTS ##
#############
import numpy as np
from sklearn.metrics import accuracy_score
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms


In [ ]:
######################
## DEVICE SELECTION ##
######################
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

###########################
## IMPORT MNIST DATA SET ##
############################
# We will only use the digit 0 for training, and then test on all digits to see how 
# well the autoencoder can reconstruct them.
# Set the batch size and define the transformation to convert images to tensors.
batch_size = 128
transform = transforms.ToTensor()

# Load the MNIST training dataset with the specified transformation. Note that I have set
# set the root folder to be outside of the repository to avoid issues with git. You may need 
# to change this path.
train_dataset = datasets.MNIST(
    root="../data",
    train=True,
    download=True,
    transform=transform
)

# Load the MNIST test dataset with the specified transformation. Note that I have set
# set the root folder to be outside of the repository to avoid issues with git. You may need 
# to change this path.
test_dataset = datasets.MNIST(
    root="../data",
    train=False,
    download=True,
    transform=transform
)


# Keep only digit 0 for training. The test dataset will contain all digits to evaluate how well 
# the autoencoder can determine anomolies.
indices = [i for i, (x, y) in enumerate(train_dataset) if y == 0]
train_subset = Subset(train_dataset, indices)

# Create a DataLoader for the training subset with the specified batch size and shuffling enabled.
train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

In [ ]:
#######################
## AUTOENCODER MODEL ##
#######################
# Define the architecture of the autoencoder. The encoder compresses the input from 784 
# dimensions (28x28 pixels) to a latent space of 32 dimensions, while the decoder reconstructs 
# the input back to 784 dimensions. The ReLU activation function is used in the hidden layers, 
# and a Sigmoid activation function is used in the output layer to ensure that the reconstructed 
# pixel values are between 0 and 1.
class Autoencoder(nn.Module):
    def __init__(self):
        """
        Inputs:
            None.
        Returns:
            None.
        Initializes the autoencoder model with an encoder and decoder architecture.
        """
        # Call the parent class constructor to initialize the nn.Module.
        super().__init__()

        # Define the encoder as a sequential model with two linear layers and ReLU activation.
        self.encoder = nn.Sequential(
            nn.Linear(784, 128),
            nn.ReLU(),
            nn.Linear(128, 32)
        )

        # Define the decoder as a sequential model with two linear layers and ReLU activation.
        self.decoder = nn.Sequential(
            nn.Linear(32, 128),
            nn.ReLU(),
            nn.Linear(128, 784),
            nn.Sigmoid()
        )

    def forward(self, x):
        """
        Inputs:
            x: Input tensor of shape (batch_size, 784).
        Returns:
            x_hat: Reconstructed tensor of shape (batch_size, 784).
        Defines the forward pass of the autoencoder, where the input is passed through the encoder
        to obtain the latent representation, and then through the decoder to reconstruct the input.
        """
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat

In [33]:
##############################
## TRAINING THE AUTOENCODER ##
##############################
# Initialize the autoencoder model and move it to the selected device (GPU or CPU). Set up the
# optimizer (Adam) and the loss function (Mean Squared Error). Train the model for a specified 
# number of epochs, where in each epoch we iterate through the training data, perform a forward
# pass, compute the loss, and update the model parameters.
model = Autoencoder().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

epochs = 200

for epoch in range(epochs):
    total_loss = 0
    model.train()

    # Iterate through the training data in batches. For each batch, reshape the input 
    # images to be vectors of size 784, move them to the selected device, and perform 
    # the forward pass through the model. Compute the loss between the reconstructed output 
    # and the original input, perform backpropagation, and update the model parameters.

    # Note that we only use the x data here since the model is unsupervised. 
    for x, _ in train_loader:
        x = x.view(-1, 784).to(device) # Reshape the input images to be vectors of size 784 
                                       # and move to device

        optimizer.zero_grad()
        x_hat = model(x)
        loss = criterion(x_hat, x)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    if (epoch+1) % 10 == 0:
        print("Epoch:", epoch+1, "Loss:", total_loss/len(train_loader))

Epoch: 10 Loss: 0.018972893384225826
Epoch: 20 Loss: 0.012729487580029255
Epoch: 30 Loss: 0.010483087951991153
Epoch: 40 Loss: 0.009058189221677627
Epoch: 50 Loss: 0.007940321794136408
Epoch: 60 Loss: 0.007240376544204798
Epoch: 70 Loss: 0.006747705782664583
Epoch: 80 Loss: 0.006347949636426378
Epoch: 90 Loss: 0.00606238228724675
Epoch: 100 Loss: 0.0057819627542445
Epoch: 110 Loss: 0.005572312897903488
Epoch: 120 Loss: 0.0053638969647123455
Epoch: 130 Loss: 0.005153139537953316
Epoch: 140 Loss: 0.005042759929486411
Epoch: 150 Loss: 0.004882469892184785
Epoch: 160 Loss: 0.004785040185410291
Epoch: 170 Loss: 0.004671795977319175
Epoch: 180 Loss: 0.0045562541766211075
Epoch: 190 Loss: 0.004495616942802642
Epoch: 200 Loss: 0.004411207204882769


In [35]:
#############################
## TESTING THE AUTOENCODER ##
#############################
model.eval()
errors = []
labels = []

with torch.no_grad():
    for x, y in test_loader:
        x = x.view(-1, 784).to(device)
        x_hat = model(x)

        error = F.mse_loss(x_hat, x, reduction='sum').item()
        errors.append(error)
        labels.append(y.item())

In [36]:
##########################################
## EVALUATING THE RECONSTRUCTION ERRORS ##
##########################################
# Determine the average reconstruction error if the label is 0 and the average reconstruction 
# error if the label is not 0. We expect the error to be much higher for non-zero digits since 
# the model was only trained on zeros.

zeros_labels = np.where(np.array(labels) == 0)[0]
zeros_errors = np.array(errors)[zeros_labels].mean()
print("Zero Reconstruction Error:", zeros_errors)

nonzeros_labels = np.where(np.array(labels) != 0)[0]
nonzeros_errors = np.array(errors)[nonzeros_labels].mean()
print("Non-Zero Reconstruction Error:", nonzeros_errors)

Zero Reconstruction Error: 4.263284396212928
Non-Zero Reconstruction Error: 21.410146391259595


In [ ]:
#########################
## DETERMINE ANOMALIES ##
#########################

# Set a threshold for anomaly detection based on the mean and standard deviation of the 
# reconstruction errors. Here, we set the threshold to be the mean reconstruction error of the 
# zeros plus 0.5 times the standard deviation of all errors. However, this is an arbitrary choice 
# and you may want to experiment with different thresholds to see how it affects the classification 
# performance.
threshold = np.mean(zeros_errors) + 0.5*np.std(errors)

# Classify each test sample as an anomaly (1) if its reconstruction error is greater than the 
# threshold, or as normal (0) otherwise.
predictions = [1 if e > threshold else 0 for e in errors]

labels_binary = [1 if l != 0 else 0 for l in labels]

accuracy_score(labels_binary, predictions)*100

98.5

## Resources
* [An Introduction to Autoencoders](https://arxiv.org/abs/2201.03898)
* [Tutorial 1: Intro to Autoencoders (Colab Notebook)](https://compneuro.neuromatch.io/tutorials/Bonus_Autoencoders/student/Bonus_Tutorial1.html)
* [What is an autoencoder?](https://www.ibm.com/think/topics/autoencoder)
* [Autoencoders](http://ufldl.stanford.edu/tutorial/unsupervised/Autoencoders/)
* [Auto-Encoder: What Is It? And What Is It Used For? (Part 1)](https://towardsdatascience.com/auto-encoder-what-is-it-and-what-is-it-used-for-part-1-3e5c6f017726/)
* [Autoencoders](https://medium.com/@divakar1591/autoencoders-6fab1a9a9f9c)
* [Introduction to autoencoders.](https://www.jeremyjordan.me/autoencoders/)
* [Autoencoders | Deep Learning Animated (Video)](https://www.youtube.com/watch?v=hZ4a4NgM3u0)


In [5]:
from torch_geometric.datasets import Planetoid
dataset = Planetoid(root="data/Planetoid", name="PubMed")

Processing...
Done!
